# Robust IoT Botnet Detection Under Incomplete Network Traffic and Cross-Device Distribution Shifts
**Part 1: Data + Random Forest baseline + missing-data simulation** (BRB model comes in Part 2)

Cell gulo upor theke nicher dike ek ek kore Run koro (Shift+Enter). Kono cell-e error ashle oi error-ta ami-ke paste koro.

Experiment design:
- **Source device** = jekhane train korbo, **Target device** = jekhane shudhu test korbo (cross-device shift)
- **Missing data** = shudhu *test* data-te NaN boshabo (deployment-e traffic incomplete hoy, training lab-e clean)
- Baseline = Random Forest + median imputation

## 1. Setup ar dataset download

In [ ]:
!pip install -q kagglehub
import kagglehub, glob, os
path = kagglehub.dataset_download("mkashifn/nbaiot-dataset")
print("Dataset path:", path)
for f in sorted(glob.glob(path + "/**/*.csv", recursive=True)):
    print(os.path.basename(f), round(os.path.getsize(f)/1e6, 1), "MB")

## 2. Device list dekho
`device_info.csv` te bola ache kon number-e kon device. Niche print hobe.

In [ ]:
import pandas as pd
info_files = glob.glob(path + "/**/device_info.csv", recursive=True)
if info_files:
    print(pd.read_csv(info_files[0]).to_string())
else:
    print("device_info.csv paini, file list upore dekho")

## 3. Config (eta change kore onno device pair-o try korte paro)

In [ ]:
SOURCE = 1        # train device number
TARGET = 2        # test-only device number
N_PER_CLASS = 20000   # proti class theke koyta row (RAM bachate)
SEED = 42
N_FEATURES = 6    # BRB-r jonno kom feature (Part 2-te lagbe)
MISSING_RATES = [0.0, 0.1, 0.3, 0.5]
N_REPEATS = 3     # random mask koyebar repeat kore average

## 4. Data load (benign = 0, Mirai attack = 1)

In [ ]:
import numpy as np, warnings
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)

def find_files(dev, kind):
    return sorted(glob.glob(f"{path}/**/{dev}.{kind}*.csv", recursive=True))

def load_device(dev, n_per_class=N_PER_CLASS, seed=SEED):
    benign_files = find_files(dev, "benign")
    mirai_files = find_files(dev, "mirai")
    assert benign_files, f"Device {dev}-er benign file paini. Upore file list dekho."
    assert mirai_files, f"Device {dev}-er mirai file paini."
    benign = pd.concat([pd.read_csv(f).astype("float32") for f in benign_files], ignore_index=True)
    # proti mirai file (scan, ack, syn, udp, udpplain) theke soman soman nao
    per_file = max(1, n_per_class // len(mirai_files))
    parts = []
    for f in mirai_files:
        d = pd.read_csv(f).astype("float32")
        parts.append(d.sample(min(len(d), per_file), random_state=seed))
    mirai = pd.concat(parts, ignore_index=True)
    benign = benign.sample(min(len(benign), n_per_class), random_state=seed)
    benign = benign.assign(label=0)
    mirai = mirai.assign(label=1)
    df = pd.concat([benign, mirai], ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)
    return df

A = load_device(SOURCE)
B = load_device(TARGET)
print("Source:", A.shape, A.label.value_counts().to_dict())
print("Target:", B.shape, B.label.value_counts().to_dict())

## 5. Domain shift ta dekho
Same feature, duto device-e distribution koto alada? (Eta paper-er motivation figure hote pare)

In [ ]:
import matplotlib.pyplot as plt
feat_cols = [c for c in A.columns if c != "label"]
demo = feat_cols[0]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for k, (name, d) in enumerate([("Source", A), ("Target", B)]):
    for lab, nm in [(0, "benign"), (1, "mirai")]:
        ax[k].hist(np.log1p(d.loc[d.label == lab, demo].clip(lower=0)), bins=40, alpha=0.5, label=nm)
    ax[k].set_title(f"{name} device, feature: {demo}")
    ax[k].legend()
plt.tight_layout(); plt.show()

## 6. Source data split + feature selection (shudhu source TRAIN diye)
Test data theke feature select korle data leakage hoy, tai shudhu train part use hocche. Correlated feature bad diye top-K nicchi.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

Xa, ya = A[feat_cols], A["label"]
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(Xa, ya, test_size=0.3, stratify=ya, random_state=SEED)
Xb, yb = B[feat_cols], B["label"]

rf_sel = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1).fit(Xa_tr, ya_tr)
imp = pd.Series(rf_sel.feature_importances_, index=feat_cols).sort_values(ascending=False)

corr = Xa_tr.corr().abs()
selected = []
for f in imp.index:
    if all(corr.loc[f, s] < 0.9 for s in selected):
        selected.append(f)
    if len(selected) == N_FEATURES:
        break
print("Selected features:", selected)
imp.head(15).plot.barh(figsize=(6, 4)); plt.gca().invert_yaxis(); plt.title("Top feature importance (source train)"); plt.show()

## 7. Baseline: Random Forest (clean data)
Duto scenario: **A->A** (same device) ar **A->B** (cross-device).

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, precision_score, recall_score

def make_rf(feats):
    return make_pipeline(SimpleImputer(strategy="median"),
                         RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1))

def scores(y, p):
    return dict(F1=f1_score(y, p), Precision=precision_score(y, p, zero_division=0), Recall=recall_score(y, p))

def run_baseline(feats):
    m = make_rf(feats).fit(Xa_tr[feats], ya_tr)
    return m, {"A->A": scores(ya_te, m.predict(Xa_te[feats])),
               "A->B": scores(yb, m.predict(Xb[feats]))}

m_all, res_all = run_baseline(feat_cols)
m_sel, res_sel = run_baseline(selected)
print(pd.DataFrame({"all 115 feats": {k: v["F1"] for k, v in res_all.items()},
                    f"selected {N_FEATURES} feats": {k: v["F1"] for k, v in res_sel.items()}}).round(4))
print()
print("Selected-feature details:"); print(pd.DataFrame(res_sel).round(4))

## 8. Missing data simulate
Duto pattern:
- **random** = packet loss/sampling (jekono cell random missing)
- **asymmetric** = routing-er karone ekta pura feature-group (jemon ekta direction-er shob feature) missing

In [ ]:
def mask_random(X, rate, rng):
    X = X.copy()
    M = rng.random(X.shape) < rate
    return X.mask(M)

def mask_asymmetric(X, rate, rng):
    # rate fraction feature column purapuri missing (ek side dekha jachche na)
    X = X.copy()
    k = int(round(rate * X.shape[1]))
    cols = rng.choice(X.columns, size=k, replace=False)
    X[cols] = np.nan
    return X

demo = mask_random(Xb[selected].head(5), 0.3, np.random.default_rng(0))
print("30% random missing example:"); display(demo.round(3))

## 9. Random Forest degrade hocche koto? (main baseline experiment)

In [ ]:
def evaluate(model, feats, masker, rates, n_repeats=N_REPEATS):
    rows = []
    for rate in rates:
        for scen, (X, y) in {"A->A": (Xa_te, ya_te), "A->B": (Xb, yb)}.items():
            f1s = []
            for r in range(n_repeats if rate > 0 else 1):
                rng = np.random.default_rng(1000 + r)
                Xm = masker(X[feats], rate, rng) if rate > 0 else X[feats]
                f1s.append(f1_score(y, model.predict(Xm)))
            rows.append(dict(Scenario=scen, MissingRate=rate, F1=np.mean(f1s), Std=np.std(f1s)))
    return pd.DataFrame(rows)

m_sel_fit = make_rf(selected).fit(Xa_tr[selected], ya_tr)
res_rand = evaluate(m_sel_fit, selected, mask_random, MISSING_RATES)
res_asym = evaluate(m_sel_fit, selected, mask_asymmetric, MISSING_RATES)

print("RANDOM missing:");     print(res_rand.pivot(index="MissingRate", columns="Scenario", values="F1").round(4))
print("\nASYMMETRIC missing:"); print(res_asym.pivot(index="MissingRate", columns="Scenario", values="F1").round(4))

## 10. Graph

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for a, (title, res) in zip(ax, [("Random missing", res_rand), ("Asymmetric missing", res_asym)]):
    for scen in ["A->A", "A->B"]:
        d = res[res.Scenario == scen]
        a.errorbar(d.MissingRate * 100, d.F1, yerr=d.Std, marker="o", label=f"RF {scen}")
    a.set_title(title); a.set_xlabel("Missing rate (%)"); a.grid(alpha=0.3); a.legend()
ax[0].set_ylabel("F1-score")
plt.tight_layout(); plt.savefig("rf_baseline_missing.png", dpi=200); plt.show()

## 11. Result save koro (Part 2-te BRB-r sathe compare korbo)

In [ ]:
import json
os.makedirs("results", exist_ok=True)
res_rand.assign(Pattern="random").to_csv("results/rf_random.csv", index=False)
res_asym.assign(Pattern="asymmetric").to_csv("results/rf_asym.csv", index=False)
json.dump({"source": SOURCE, "target": TARGET, "selected_features": selected},
          open("results/config.json", "w"), indent=2)
print("Saved: results/ folder + rf_baseline_missing.png")
print("Selected features (Part 2-er jonno):", selected)

---
## Ekhon ki?
1. Upore-r shob output (table + graph + selected features list) ami-ke pathao ba screenshot dao.
2. Tarpor **Part 2: BRB engine** baniye debo, ei same selected feature, same split, same missing mask diye. Tahole comparison fair hobe.